# Preprocessing

This script performs several preprocessing steps to prepare the data for using them to finetune an LLM (e.g., GottBERT). Specifically, this script removes short speeches, removes introductions and conclusions from all speeches that were kept, and then divides the speeches into smaller sections. Next, the data are augmented to take into account that some parties are underrepresented. Finally, a dataset is created in which party names are removed from the speeches so the model cannot use this information.

## Preparation

In [1]:
import pandas as pd
import numpy as np
import random
import math
from datetime import datetime
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
path_to_data = '/content/drive/MyDrive/TL/data/'
df = pd.read_parquet(path_to_data + 'speeches_clean.parquet')

In [4]:
df.head()

,legislative_period,date,speaker_id,first_name,last_name,role,party,speech
0,19,15.01.2020,11004699,Astrid,Damerow,keine,CDU/CSU,Frau Präsidentin! Verehrte Kolleginnen und Kol...
1,19,15.01.2020,11004393,Johann,Saathoff,keine,SPD,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...
2,19,15.01.2020,11003706,Artur,Auernhammer,keine,CDU/CSU,Verehrte Frau Präsidentin! Liebe Kolleginnen u...
3,19,15.01.2020,11003604,Friedrich,Ostendorff,keine,Die Grünen,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...
4,19,15.01.2020,11003740,Heidrun,Bluhm-Förster,keine,Die Linke,Frau Präsidentin! Verehrte Kolleginnen und Kol...


## Labelling

First, we make numeric labels:

In [5]:
df['labels'] = df['party']
parties = df['labels'].unique()
mapping = {party: i for i, party in enumerate(parties)}
df['labels'] = df['labels'].map(mapping)

We save the mapping:

In [6]:
with open(path_to_data + 'speches_mapping.pkl', 'wb') as fp:
    pickle.dump(mapping, fp)

## Remove short speeches

We only keep speeches with at least 200 words.

In [7]:
df['no_words'] = df['speech'].apply(lambda x: len(x.split())).astype("int16")
df = df[df['no_words'] >= 200]

## Remove introductions and conclusions

We define a function that removes introductions and conclusions (e.g., the part where the speaker says hello and goodbye). We use full stops and exclamation marks as indicators for the end/start of the first/last sentence.

In [8]:
def strip_speech(speech):

  # find first split point:
  first_fs = speech.find(".") # first full stop
  first_em = speech.find("!") # first exclamation mark

  if first_em == -1:
    first_em = len(speech)+10 # if there is no exclamation mark, set this to a large number so -1 is not considered as a split point

  if first_fs < first_em:
    first_sp = first_fs
  else:
    first_sp = first_em

  second_fs = speech.find(".", first_sp+1) # second full stop
  second_em = speech.find("!", first_sp+1) # second exclamation mark

  if second_em == -1:
    second_em = len(speech)+10 # if there is no exclamation mark, set this to a large number so -1 is not considered as a split point

  if second_fs < second_em:
    second_sp = second_fs
  else:
    second_sp = second_em

  # find last split point:
  last_fs = speech.rfind(".")
  last_em = speech.rfind("!")

  if last_fs > last_em:
    last_sp = last_fs
  else:
    last_sp = last_em

  ntlast_fs = speech.rfind(".",0,last_sp)
  ntlast_em = speech.rfind("!",0,last_sp)

  if ntlast_fs > ntlast_em:
    ntlast_sp = ntlast_fs
  else:
    ntlast_sp = ntlast_em

  # split speech at both points (retain everything in the middle):
  speech = speech[(second_sp+2):(ntlast_sp+1)]

  return speech

Illustrate the use of the function with a particular speech:

In [9]:
speech = df.iloc[15]['speech']
stripped_speech = strip_speech(speech)

In [10]:
print(speech) # beginning of speech before removing the introduction

Sehr geehrte Frau Präsidentin! Liebe Kolleginnen und Kollegen! Auch wir Freie Demokraten sind selbstverständlich für ein höchstmögliches Maß an Sicherheit im Luftverkehr. Wir finden es auch richtig - wie es in dem Vorschlag beschrieben wird und allgemein im Bereich der Luftsicherheit gilt -, dass Unternehmen einschließlich Luftfahrtunternehmen, Betreiber von Flughäfen, Dienstleister an Flughäfen, Verkehrspiloten und Berufspiloten luftsicherheitsrechtlichen Zuverlässigkeitsüberprüfungen unterzogen werden. Aber mit einer Zuverlässigkeitsüberprüfung ohne Differenzierung zwischen der Reinigungskraft und dem Verkehrspiloten in jedem Ort, im Flughafen oder auf einem kleinen Segelfluggelände - denn der Motorsegler ist auch davon betroffen; das gilt auch für den Segelflieger, der eine Motorsegelflugberechtigung hat -, wird alles über einen Kamm geschert. Selbstverständlich macht das einen Riesenunterschied. Sie machen aber weiterhin dabei keinen Unterschied, und das ist nicht in Ordnung. Ich k

In [11]:
print(stripped_speech) # beginning of the speech after removing the introduction

Auch wir Freie Demokraten sind selbstverständlich für ein höchstmögliches Maß an Sicherheit im Luftverkehr. Wir finden es auch richtig - wie es in dem Vorschlag beschrieben wird und allgemein im Bereich der Luftsicherheit gilt -, dass Unternehmen einschließlich Luftfahrtunternehmen, Betreiber von Flughäfen, Dienstleister an Flughäfen, Verkehrspiloten und Berufspiloten luftsicherheitsrechtlichen Zuverlässigkeitsüberprüfungen unterzogen werden. Aber mit einer Zuverlässigkeitsüberprüfung ohne Differenzierung zwischen der Reinigungskraft und dem Verkehrspiloten in jedem Ort, im Flughafen oder auf einem kleinen Segelfluggelände - denn der Motorsegler ist auch davon betroffen; das gilt auch für den Segelflieger, der eine Motorsegelflugberechtigung hat -, wird alles über einen Kamm geschert. Selbstverständlich macht das einen Riesenunterschied. Sie machen aber weiterhin dabei keinen Unterschied, und das ist nicht in Ordnung. Ich kann es eigentlich auch nicht fassen, dass immer noch von einer 

In [12]:
print(speech[(len(speech)-100):]) # end of the speech before removing the conclusion

tag Vernunft an und schaffen die ZÜP für die Privatpiloten und die Luftsportler ab. Herzlichen Dank.


In [13]:
print(stripped_speech[(len(stripped_speech)-100):]) # end of the speech after removing the introdcution

 Deutschen Bundestag Vernunft an und schaffen die ZÜP für die Privatpiloten und die Luftsportler ab.


Apply the function to the whole data set:

In [14]:
df['speech_stripped'] = df['speech'].apply(lambda s: strip_speech(s))

And again remove the speeches that are too short:

In [15]:
df['no_words_stripped'] = df['speech_stripped'].apply(lambda x: len(x.split())).astype("int16")
df = df[df['no_words_stripped'] >= 200]

## Divide the speeches into smaller parts

Here, we divide the speeches into smaller parts that roughly match the number of tokens that the model can process (512). For German text, this is the case for snippets of about 3000 characters. We define a function to divide the speeches into snippets of 3000 characters (at the same time, we keep the lower word limit of 200).

In [16]:
def divide_speech(speech, string_size, word_tolerance):
  parts = math.ceil(len(speech)/string_size)
  if parts == 1:  # if the speech is short enough, simply return it
      return [speech]

  order = ["begin", "end"]
  this_order = random.sample(order, 1)
  divided_speech = []

  for i in range(parts):
      if this_order == "begin":
          part = speech[(i*string_size):(i+1)*string_size]
      else:  # this_order <sup> </sup> == "end"
          if (len(speech)-(i+1)*string_size) < 0:
              part = speech[0:(len(speech)-i*string_size)]
          else:
              part = speech[(len(speech)-(i+1)*string_size):(len(speech)-i*string_size)]

      # try to sample whole sentences where possible
      start_idx = part.find(".")
      end_idx = part.rfind(".")
      start = start_idx + 2 if start_idx != -1 else 0
      end = end_idx + 1 if end_idx != -1 else len(part)

      # keep edges untrimmed for the outermost slice depending on direction
      if (i == 0 and this_order == "begin") or (i == (parts - 1) and this_order == "end"):
          start = 0
      if (i == 0 and this_order == "end") or (i == (parts - 1) and this_order == "begin"):
          end = len(part)

      part = part[start:end]

      # word tolerance
      if len(part.split()) >= word_tolerance:
          divided_speech.append(part)

  random.shuffle(divided_speech)
  return divided_speech

Apply the function to the data:

In [17]:
df_snippets = pd.DataFrame()

for s in range(0, len(df)):
  split_speech = divide_speech(df.iloc[s]['speech_stripped'], string_size = 3000, word_tolerance = 200)
  if len(split_speech) > 0:
    df_row = pd.DataFrame(df.iloc[s])
    df_rows = pd.concat([df_row.T]*len(split_speech))
    df_rows['speech_stripped'] = split_speech
    df_snippets = pd.concat([df_snippets, df_rows])

We check the distributions of the character and word counts for the different parties that result:

In [18]:
df_snippets['no_chars'] = df_snippets['speech_stripped'].apply(len).astype("int16")
df_snippets['no_words'] = df_snippets['speech_stripped'].apply(lambda x: len(x.split())).astype("int16")

In [19]:
df_snippets.groupby('party')['no_chars'].describe()

,count,mean,std,min,25%,50%,75%,max
party,,,,,,,,
AfD,4264.0,2696.774625,398.826845,1192.0,2650.00,2873.0,2946.0,3000.0
CDU/CSU,10847.0,2619.700655,468.258233,1252.0,2465.50,2853.0,2938.0,3000.0
Die Grünen,5588.0,2648.615963,432.699238,1237.0,2539.75,2852.0,2938.0,3000.0
Die Linke,2878.0,2612.965949,432.608081,1218.0,2402.00,2816.0,2932.0,3000.0
FDP,4841.0,2633.933072,459.273153,1211.0,2478.00,2862.0,2940.0,2998.0
SPD,9811.0,2649.330547,457.815453,1177.0,2586.50,2867.0,2943.0,3000.0


In [20]:
df_snippets.groupby('party')['no_words'].describe()

,count,mean,std,min,25%,50%,75%,max
party,,,,,,,,
AfD,4264.0,384.097561,58.530839,200.0,368.0,401.0,422.0,486.0
CDU/CSU,10847.0,381.285148,69.308300,200.0,355.0,407.0,429.0,517.0
Die Grünen,5588.0,384.079814,64.032073,200.0,361.0,405.0,428.0,490.0
Die Linke,2878.0,375.176511,64.176921,200.0,342.0,396.0,422.0,499.0
FDP,4841.0,381.200578,67.827014,200.0,356.0,405.0,427.0,501.0
SPD,9811.0,384.288350,67.551971,200.0,367.5,408.0,429.0,501.0


We notice that the number of speeches per party is unbalanced across parties. Therefore, we add additional snippets from the middle parts of the speeches for the underrepresented parties.

## Add samples for underrepresented parties

In [21]:
def sample_middle_snippet(speech, string_size, word_tolerance, jitter=0.25):

  # window around the middle of the speech, with small random shift
  if len(speech) <= string_size:
    window = speech
  else:
    center = len(speech) // 2
    max_shift = int(string_size * jitter)
    shift = random.randint(-max_shift, max_shift) if max_shift > 0 else 0
    start = max(0, min(len(speech) - string_size, center - string_size // 2 + shift))
    end = start + string_size
    window = speech[start:end]

  # try to align to sentence boundaries inside the window
  start_idx = window.find(".")
  end_idx = window.rfind(".")
  start = start_idx + 2 if start_idx != -1 else 0
  end = end_idx + 1 if end_idx != -1 else len(window)
  part = window[start:end]

  if len(part.split()) >= word_tolerance:
      return part

  # return None if cannot satisfy tolerance; caller will fall back
  return None


In [22]:
def balance_snippets_by_party(df_base, df_snips, text_col='speech_stripped', party_col='party',
  string_size=3000, word_tolerance=200, random_state=None):
  if random_state is not None:
    random.seed(random_state)

  counts = df_snips[party_col].value_counts()
  if counts.empty:
      return df_snips.copy()

  target = counts.max()
  balanced = df_snips.copy()

  for party, count in counts.items():
      deficit = target - count
      if deficit <= 0:
          continue

      df_party_speeches = df_base[df_base[party_col] == party]
      if df_party_speeches.empty:
          # no source speeches; just duplicate existing party snippets
          party_snips = balanced[balanced[party_col] == party]
          if not party_snips.empty:
              dup = party_snips.sample(n=deficit, replace=True, random_state=random_state)
              balanced = pd.concat([balanced, dup], ignore_index=True)
          continue

      added_rows = []
      attempts = 0
      max_attempts = deficit * 10  # avoid infinite loops

      while deficit > 0 and attempts < max_attempts:
          attempts += 1
          src = df_party_speeches.sample(n=1, replace=True, random_state=None).iloc[0]
          speech = src[text_col]

          middle = sample_middle_snippet(speech, string_size, word_tolerance)
          if middle is None:
              # fallback: any valid snippet from this speech
              candidates = divide_speech(speech, string_size, word_tolerance)
              if len(candidates) > 0:
                  middle = random.choice(candidates)
              else:
                  # last resort: duplicate any existing snippet from this party
                  party_cands = balanced.loc[balanced[party_col] == party, text_col]
                  if len(party_cands) == 0:
                      continue
                  middle = party_cands.sample(1).iloc [0]

          row = src.to_dict()
          row[text_col] = middle
          added_rows.append(row)
          deficit -= 1

      if added_rows:
          balanced = pd.concat([balanced, pd.DataFrame(added_rows)], ignore_index=True)

  # Optional: shuffle rows
  balanced = balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)
  return balanced


In [23]:
df_snippets = balance_snippets_by_party(
    df_base=df,
    df_snips=df_snippets,
    text_col='speech_stripped',
    party_col='party',
    string_size=3000,
    word_tolerance=200,
    random_state=42
)

In [24]:
df_snippets['no_chars'] = df_snippets['speech_stripped'].apply(len).astype("int16")
df_snippets['no_words'] = df_snippets['speech_stripped'].apply(lambda x: len(x.split())).astype("int16")

In [25]:
df_snippets.groupby('party')['no_chars'].describe()

,count,mean,std,min,25%,50%,75%,max
party,,,,,,,,
AfD,10847.0,2696.290311,353.505444,1192.0,2649.0,2827.0,2918.0,3000.0
CDU/CSU,10847.0,2619.700655,468.258233,1252.0,2465.5,2853.0,2938.0,3000.0
Die Grünen,10847.0,2663.482898,385.722729,1198.0,2586.5,2818.0,2915.0,3000.0
Die Linke,10847.0,2582.119111,410.996267,1218.0,2423.5,2747.0,2882.0,3000.0
FDP,10847.0,2673.921361,388.571251,1211.0,2634.0,2830.0,2916.0,2998.0
SPD,10847.0,2658.084908,445.387309,1177.0,2626.0,2860.0,2939.0,3000.0


In [26]:
df_snippets.groupby('party')['no_words'].describe()

,count,mean,std,min,25%,50%,75%,max
party,,,,,,,,
AfD,10847.0,383.548723,52.643765,200.0,367.0,397.0,417.0,486.0
CDU/CSU,10847.0,381.285148,69.308300,200.0,355.0,407.0,429.0,517.0
Die Grünen,10847.0,386.070803,57.930083,200.0,368.0,402.0,424.0,491.0
Die Linke,10847.0,370.285793,61.695820,200.0,341.0,388.0,415.0,499.0
FDP,10847.0,386.394210,58.393336,200.0,370.0,403.0,424.0,512.0
SPD,10847.0,385.354292,65.820548,200.0,370.0,407.0,428.0,501.0


## Remove party names from speeches

Here, we remove party names from the speeches so that the LLM can not use information on their frequency (which could be problematic because it might be the case that speakers refer to their own party more frequently compared to other parties). When doing so, we have to protect certain phrases which contain party names (e.g., 'Europäische Union' which contains 'Union, 'Grüne Woche' which contains 'Grüne). We therefore first replace these phrases by placeholders and revert them after party information was omitted.

In [27]:
df_snippets_noparty = df_snippets.copy()

In [28]:
def change_multiple_strings(text, dic):
    for i, j in dic.items():
        text = text.replace(i, j)
    return text

change_these  = {"Europäischen Union": "fill_word_1", "Europäische Union": "fill_word_2",
                  "europäischen Union": "fill_word_3", "europäische Union": "fill_word_4",
                  "Grünen Woche" : "fill_word_5", "Grüne Woche" : "fill_word_6"}

reverse_these  = {"fill_word_1": "Europäischen Union", "fill_word_2": "Europäische Union",
                  "fill_word_3": "europäischen Union", "fill_word_4": "europäische Union",
                  "fill_word_5": "Grünen Woche",  "fill_word_6": "Grüne Woche"}

In [29]:
df_snippets_noparty['speech_stripped'] = df_snippets_noparty['speech_stripped'].apply(lambda s: change_multiple_strings(s, change_these))

In [30]:
def remove_substrings_replace(speech, substrings, replacement):
    for substring in substrings:
        speech = speech.replace(substring, replacement)
    return speech

strings_to_remove = ["FDP", "Freie Demokratische Partei",  "Freien Demokratischen Partei", "Freie-Demokratische Partei",  "Freien-Demokratischen Partei",
              "Freien Demokraten", "Freie Demokraten",  "Freie Demokratin", "Freier Demokrat", "Liberalen", "Liberaler", "Liberale",

              "SPD", "Sozialdemokratische Partei Deutschlands", "Sozialdemokratischen Partei Deutschlands",
              "Sozialdemokratische Partei", "Sozialdemokratischen Partei", "Sozial-Demokratische Partei", "Sozial-Demokratischen Partei",
               "Sozial-demokratische Partei", "Sozial-demokratischen Partei", "Sozialdemokraten", "Sozialdemokratin", "Sozialdemokrat",

              "CDU/CSU",
              "CDU", "Christlich Demokratische Union", "Christlich Demokratischen Union", "Christlich-Demokratische Union",
              "Christlich-Demokratischen Union", "Christlich-demokratische Union", "Christlich-demokratischen Union",

              "CSU", "Christlich-Soziale Union", "Christlich-Sozialen Union", "Christlich Soziale Union", "Christlich Sozialen Union",
              "Christlich-soziale Union", "Christlich-sozialen Union", "Union",
              "Christdemokraten" ,  "Christdemokratin" , "Christdemokrat",

              "Bündnis 90/Die Grünen", "Bündnis 90", "Grünen", "Grüner", "Grüne",

              "Linken", "Linker", "Linke",

              "AFD", "AfD",  "Alternativen für Deutschland", "Alternative für Deutschland"]

In [31]:
df_snippets_noparty['speech_stripped'] = df_snippets_noparty['speech_stripped'].apply(lambda s: remove_substrings_replace(s, substrings = strings_to_remove, replacement = "Partei"))

In [32]:
df_snippets_noparty['speech_stripped'] = df_snippets_noparty['speech_stripped'].apply(lambda s: change_multiple_strings(s, reverse_these))

Test the result:

In [33]:
print(df_snippets.iloc[15]['speech_stripped'])

Herr Merz, ich kann es Ihnen nicht ersparen: Ihre chronische Amnesie bezüglich der 16 Jahre Angela Merkel mag ja für Sie persönlich mental wichtig sein, hat mit Realpolitik aber nichts zu tun. Wir räumen auf, was Sie ignorieren. Wir räumen 16 Jahre CDU/CSU im Bundesverteidigungsministerium auf. Ich bin überzeugt, liebe Kolleginnen und Kollegen: Das Sondervermögen und die damit verbundenen Investitionen in die Bundeswehr markieren ohne Wenn und Aber eine Zeitenwende. Mit diesem Geld konnten wir erstmals Strukturen und Grundlagen schaffen, die nötig sind, um die ungelösten Probleme anzugehen, die wir von Ihnen geerbt haben. Dennoch möchte ich an dieser Stelle ganz besonders auch der Union danken, dass wir für die Absicherung das "Sondervermögen Bundeswehr" auch ins Grundgesetz geschrieben haben. Ohne Ihre Mithilfe wäre das nicht möglich gewesen. Wir haben dank dieses Sondervermögens Investitionen in kurzer Zeit tätigen können, die seit mindestens 20 Jahren überfällig waren. Hier geht es 

In [34]:
print(df_snippets_noparty.iloc[15]['speech_stripped'])

Herr Merz, ich kann es Ihnen nicht ersparen: Ihre chronische Amnesie bezüglich der 16 Jahre Angela Merkel mag ja für Sie persönlich mental wichtig sein, hat mit Realpolitik aber nichts zu tun. Wir räumen auf, was Sie ignorieren. Wir räumen 16 Jahre Partei im Bundesverteidigungsministerium auf. Ich bin überzeugt, liebe Kolleginnen und Kollegen: Das Sondervermögen und die damit verbundenen Investitionen in die Bundeswehr markieren ohne Wenn und Aber eine Zeitenwende. Mit diesem Geld konnten wir erstmals Strukturen und Grundlagen schaffen, die nötig sind, um die ungelösten Probleme anzugehen, die wir von Ihnen geerbt haben. Dennoch möchte ich an dieser Stelle ganz besonders auch der Partei danken, dass wir für die Absicherung das "Sondervermögen Bundeswehr" auch ins Grundgesetz geschrieben haben. Ohne Ihre Mithilfe wäre das nicht möglich gewesen. Wir haben dank dieses Sondervermögens Investitionen in kurzer Zeit tätigen können, die seit mindestens 20 Jahren überfällig waren. Hier geht es 

Now, finally replace the speech column by the speech_stripped column for both data frames::

In [35]:

df_snippets = df_snippets.drop('speech', axis = 1)
df_snippets.rename(columns={"speech_stripped": "speech"}, inplace=True)
df_snippets.head()


,legislative_period,date,speaker_id,first_name,last_name,role,party,labels,no_words,speech,no_words_stripped,no_chars
0,19,07.05.2020,11003790,Jan,Korte,keine,Die Linke,3,410,Vielleicht hat es was mit Ihren Auftritten und...,566,2649
1,19,07.09.2021,11003584,Gesine,Lötzsch,keine,Die Linke,3,421,Ein Virus kann doch nicht mit militärischer Lo...,813,2923
2,20,07.07.2022,11003848,Petra,Sitte,keine,Die Linke,3,352,"Nun zur Sache. Es ist einerseits erschreckend,...",363,2580
3,20,04.07.2024,11005049,Sonja,Eichwede,keine,SPD,1,446,"Auch deshalb ist es so wichtig, dass wir hier ...",697,2887
4,19,12.02.2021,11004276,Thorsten,Frei,keine,CDU/CSU,0,263,Es ist in der Tat so: Als wir uns vor knapp ei...,263,1733


In [36]:
df_snippets_noparty = df_snippets_noparty.drop('speech', axis = 1)
df_snippets_noparty.rename(columns={"speech_stripped": "speech"}, inplace=True)
df_snippets_noparty.head()

,legislative_period,date,speaker_id,first_name,last_name,role,party,labels,no_words,speech,no_words_stripped,no_chars
0,19,07.05.2020,11003790,Jan,Korte,keine,Die Linke,3,410,Vielleicht hat es was mit Ihren Auftritten und...,566,2649
1,19,07.09.2021,11003584,Gesine,Lötzsch,keine,Die Linke,3,421,Ein Virus kann doch nicht mit militärischer Lo...,813,2923
2,20,07.07.2022,11003848,Petra,Sitte,keine,Die Linke,3,352,"Nun zur Sache. Es ist einerseits erschreckend,...",363,2580
3,20,04.07.2024,11005049,Sonja,Eichwede,keine,SPD,1,446,"Auch deshalb ist es so wichtig, dass wir hier ...",697,2887
4,19,12.02.2021,11004276,Thorsten,Frei,keine,CDU/CSU,0,263,Es ist in der Tat so: Als wir uns vor knapp ei...,263,1733


Save the results:

In [37]:
df_snippets.to_parquet(path_to_data + 'speeches_prep_party.parquet') # save speech snippets with party names
df_snippets_noparty.to_parquet(path_to_data + 'speeches_prep_noparty.parquet') # save speech snippets without party names